# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 106.6 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <StringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1
import math

asia = df.loc[df['continent'] == 'Asia']
asia = asia.loc[asia['year'].isin([2002, 2007])].copy()
asia['year'] = asia['year'].astype(str)

fig = px.scatter(
    data_frame=asia, x='gdpPercap', y='lifeExp',
    color='year',
    color_discrete_map={'2002': 'lightgrey', '2007': 'steelblue'},
    hover_name='country',
    log_x=True,
    labels={'gdpPercap': 'GDP per Capita (log scale)', 'lifeExp': 'Life Expectancy', 'year': ''},
    template='plotly_white'
)

for country in asia['country'].unique():
    c = asia.loc[asia['country'] == country]
    fig.add_scatter(x=c['gdpPercap'], y=c['lifeExp'], mode='lines',
                    line=dict(color='lightgrey', width=1),
                    showlegend=False, hoverinfo='skip')

fig.update_traces(marker=dict(size=9))

row = asia.loc[(asia['country'] == 'Cambodia') & (asia['year'] == '2007')].iloc[0]
fig.add_annotation(
    x=math.log10(row['gdpPercap']), y=row['lifeExp'],
    text='Cambodia: biggest gain (+3 years)',
    showarrow=True, arrowhead=1, ax=60, ay=40,
    font=dict(color='steelblue', size=11)
)

fig.add_annotation(
    x=math.log10(600), y=80, xanchor='left',
    text='grey = 2002, blue = 2007<br>almost every country moved up and to the right',
    showarrow=False, align='left',
    font=dict(color='steelblue', size=11),
    bgcolor='white', bordercolor='steelblue', borderwidth=1, borderpad=4
)

fig.update_layout(
    title='Every Asian country lived longer in 2007 than in 2002 — most also got richer',
    font=dict(family='Arial', size=13),
    height=550,
    legend=dict(orientation='h', x=0, y=1.06)
)
fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Task 2
import math

df07 = df.loc[df['year'] == 2007]

colour_map = {'Asia': 'steelblue', 'Africa': 'lightgrey', 'Americas': 'lightgrey',
              'Europe': 'lightgrey', 'Oceania': 'lightgrey'}

fig = px.scatter(
    data_frame=df07, x='gdpPercap', y='lifeExp',
    size='pop', size_max=55,
    color='continent',
    color_discrete_map=colour_map,
    hover_name='country',
    log_x=True,
    labels={'gdpPercap': 'GDP per Capita (log scale)', 'lifeExp': 'Life Expectancy', 'pop': 'Population'},
    template='plotly_white',
    opacity=0.7
)

for country, ax, ay in [('China', -60, -45), ('India', -40, 55), ('Japan', -40, -35), ('Afghanistan', 40, 35)]:
    row = df07.loc[df07['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=country,
        showarrow=True, arrowhead=1, ax=ax, ay=ay,
        font=dict(color='steelblue', size=11)
    )

fig.add_annotation(
    x=math.log10(450), y=78, xanchor='left',
    text='Asia covers almost the whole range:<br>Afghanistan 44 years, Japan 83 years.<br>China and India are almost 40% of the world.',
    showarrow=False, align='left',
    font=dict(color='steelblue', size=11),
    bgcolor='white', bordercolor='steelblue', borderwidth=1, borderpad=4
)

fig.update_layout(
    title='Asia stretches from Afghanistan (44 years) to Japan (83, the world\'s longest) — China and India sit in the middle',
    font=dict(family='Arial', size=13),
    showlegend=False,
    height=600
)
fig.show()
